# Comparación de matrices · TFM Energía UCM

Lee `data/gold/rejilla_matrices.csv`, que va escribiendo `scripts/rejilla_matrices.py`
después de **cada** entrenamiento. Así que se puede ejecutar con la rejilla todavía
corriendo: refresca y muestra lo que haya hasta ese momento.

**El cómputo va en el script y el análisis aquí.** Una rejilla de 36 entrenamientos no debe
depender de que el editor siga abierto, y un notebook que tarda hora y media en ejecutarse
no se puede volver a mirar sin volver a esperar.

## La pregunta

Las cuatro matrices existen para contestar tres cosas:

| comparación | pregunta |
|---|---|
| `completa` vs `nucleo` | ¿sobran variables? |
| `minima` vs `nucleo` | ¿bastan las 25 mejores? |
| `moderna` vs `nucleo` | ¿estorba la crisis del gas de 2020-2022? |

Y tres arquitecturas que cubren el espectro de tamaños —SimpleRNN ~35 k, GRU ~320 k,
Conv1D+LSTM ~460 k— porque el tamaño de la matriz interactúa con el del modelo: con menos
datos, los modelos pequeños suben posiciones.

In [ ]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")
plt.rcParams["figure.figsize"] = (11, 4)
pd.set_option("display.width", 200)

REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "gold").is_dir())
CSV = REPO / "data" / "gold" / "rejilla_matrices.csv"

d = pd.read_csv(CSV)
esperado = d.matriz.nunique() * d.arquitectura.nunique() * d.semilla.nunique()
print(f"{len(d)} entrenamientos de {esperado} previstos "
      f"({len(d) / max(esperado, 1) * 100:.0f}%)")
print(f"matrices      : {sorted(d.matriz.unique())}")
print(f"arquitecturas : {sorted(d.arquitectura.unique())}")
print(f"semillas      : {sorted(d.semilla.unique())}")
if len(d) < esperado:
    print()
    print("La rejilla sigue corriendo. Vuelve a ejecutar esta celda para refrescar.")

## 1 · Media y desviación por celda

El MAE de validación es comparable entre columnas porque **las cuatro matrices validan
sobre el mismo periodo**, 2025 entero. Lo que cambia es con cuántos días entrena cada una.

In [ ]:
piv = d.pivot_table(index="arquitectura", columns="matriz", values="MAE_val",
                    aggfunc=["mean", "std"]).round(3)
display(piv)

if d.semilla.nunique() > 1:
    fig, ax = plt.subplots(figsize=(11, 4))
    for arq, g in d.groupby("arquitectura"):
        r = g.groupby("matriz")["MAE_val"].agg(["mean", "std"]).reindex(
            [m for m in ["completa", "nucleo", "minima", "moderna"] if m in set(d.matriz)])
        ax.errorbar(r.index, r["mean"], yerr=r["std"], marker="o", capsize=4, label=arq)
    ax.set_ylabel("MAE validación (€/MWh)")
    ax.set_title("Cada punto es la media de las semillas; la barra, su desviación")
    ax.legend(); ax.grid(alpha=.3)
    plt.tight_layout(); plt.show()
    print("Si las barras de dos matrices se solapan, esa diferencia NO es concluyente")
    print("con medias sueltas. Para eso está la comparación pareada de la sección 2.")

## 2 · Comparación pareada — la que decide

Comparar medias sueltas desperdicia el diseño. Las cuatro matrices se entrenan con **las
mismas semillas**, así que restando cada matriz contra `nucleo` en la **misma semilla y
arquitectura** desaparece la varianza de inicialización, que es la que ensucia todo.

Con medias sueltas hacen falta diferencias de ~0,5 para distinguir algo; pareado basta con
~0,15. Y el criterio se lee sin estadística: **si las diferencias tienen todas el mismo
signo y superan su propia dispersión, hay ganador; si cambian de signo, es ruido.**

In [ ]:
REF = "nucleo"
base = d[d.matriz == REF].set_index(["arquitectura", "semilla"])["MAE_val"]

filas = []
for m in [x for x in d.matriz.unique() if x != REF]:
    otra = d[d.matriz == m].set_index(["arquitectura", "semilla"])["MAE_val"]
    dif = (otra - base).dropna()
    if not len(dif):
        continue
    mismo = bool((dif > 0).all() or (dif < 0).all())
    filas.append({"matriz": m, "n_pares": len(dif),
                  "dif_media": round(float(dif.mean()), 3),
                  "dif_sd": round(float(dif.std(ddof=1)), 3) if len(dif) > 1 else np.nan,
                  "mismo_signo": mismo,
                  "veredicto": (f"gana {REF}" if dif.mean() > 0 else f"gana {m}")
                               if mismo and abs(dif.mean()) > 0.15 else "empate"})

if filas:
    pareada = pd.DataFrame(filas)
    display(pareada)
    print(f"`dif_media` positiva = esa matriz es PEOR que {REF}.")
    print()
    for r in pareada.itertuples():
        if r.veredicto == "empate":
            if r.matriz == "moderna":
                print(f"· {r.matriz}: EMPATE, y es un hallazgo -- consigue lo mismo con el")
                print(f"  40 % de los días. 2020-2022 no aportan.")
            elif r.matriz == "completa":
                print(f"· {r.matriz}: empate -> gana `nucleo`, mismo resultado con 113")
                print(f"  inputs en vez de 141.")
            else:
                print(f"· {r.matriz}: empate.")
        else:
            print(f"· {r.matriz}: {r.veredicto} por {abs(r.dif_media):.3f}, "
                  f"consistente en las {r.n_pares} comparaciones.")

In [ ]:
# El detalle por semilla, que es donde se ve si el signo aguanta
if len(d.semilla.unique()) > 1:
    det = (d.pivot_table(index=["arquitectura", "semilla"], columns="matriz",
                         values="MAE_val")
             .pipe(lambda x: x.sub(x[REF], axis=0))
             .drop(columns=[REF]).round(3))
    display(det)
    print("Cada fila es una comparación pareada. Columnas todas del mismo signo = señal;")
    print("columnas que cambian de signo = ruido de inicialización.")

## 3 · Qué arquitectura gana en cada matriz

Aquí está la razón de usar tres tamaños en vez de la mejor de `nucleo`: si el orden cambia
entre columnas, el tamaño de la matriz interactúa con el del modelo, y elegir arquitectura
mirando una sola matriz habría sesgado el resultado.

In [ ]:
orden = (d.groupby(["matriz", "arquitectura"])["MAE_val"].mean()
           .groupby("matriz").rank().unstack().round(1))
display(orden)

cambia = orden.nunique(axis=1).gt(1).any() if len(orden.columns) > 1 else False
if cambia:
    print("EL ORDEN CAMBIA entre matrices: el tamaño del modelo interactúa con el de la")
    print("matriz, y hay que elegir arquitectura DESPUÉS de elegir matriz.")
else:
    print("El orden se mantiene en todas: la arquitectura se puede elegir con independencia")
    print("de la matriz, lo que simplifica la decisión.")

print()
print("Y contra la persistencia, que es el listón real:")
display(d.pivot_table(index="arquitectura", columns="matriz",
                      values="vs_naive_val_%", aggfunc="mean").round(1))

## 4 · Coste en datos

`moderna` entrena con el 40 % de las filas. Si empata, no está empatando: está ganando en
eficiencia, y eso significa que los años de la crisis del gas no aportan información
utilizable.

In [ ]:
cst = (d.groupby("matriz")
         .agg(dias_train=("dias_train", "first"), canales=("canales", "first"),
              MAE_val=("MAE_val", "mean"))
         .sort_values("dias_train", ascending=False).round(3))
cst["dias_por_punto_de_MAE"] = (cst.dias_train / (cst.MAE_val.max() - cst.MAE_val + 0.01)
                                ).round(0)
display(cst[["dias_train", "canales", "MAE_val"]])

fig, ax = plt.subplots(figsize=(7, 4))
for m, r in cst.iterrows():
    ax.scatter(r.dias_train, r.MAE_val, s=120)
    ax.annotate(m, (r.dias_train, r.MAE_val), xytext=(6, 4), textcoords="offset points")
ax.set_xlabel("días de entrenamiento"); ax.set_ylabel("MAE validación (€/MWh)")
ax.set_title("Cuántos datos cuesta cada punto de error")
ax.grid(alpha=.3); plt.tight_layout(); plt.show()

## 5 · La decisión

Y el comando de la fase B con la matriz elegida.

In [ ]:
ganadora = d.groupby("matriz")["MAE_val"].mean().idxmin()
media = d.groupby("matriz")["MAE_val"].mean().round(3)

print("MAE de validación medio por matriz:")
print(media.sort_values().to_string())
print()
print(f"Mejor media: {ganadora}")
print()
print("PERO la media no decide: mira la sección 2. Si `nucleo` empata con la que tenga")
print("mejor media, gana `nucleo` por ser la referencia ya probada y auditada -- cambiar")
print("de matriz obliga a rehacer todo lo que cuelga de ella.")
print()
print("Cuando lo tengas claro, la fase B:")
print()
print(f"    python scripts/entrenar_finales.py --matriz {ganadora} "
      f"--semillas 3 --guardar-modelos")
print()
print("8 familias x 3 semillas = 24 entrenamientos, ~1,5 h. Deja los .keras con su")
print(".preprocesado.json, las predicciones de validación y el ensemble por familia.")